In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 120
})
PALETTE = ['#1B4332', '#B87333', '#D4A574', '#2D3436', '#5C8A4D']

In [ ]:
df = pd.read_csv('../data/housing.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
features = ['Area Income', 'Area House Age', 'Area No of Rooms', 'Area No of Bedrooms', 'Area Population']
for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers}")

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='YlOrBr', linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_features = ['Area Income', 'Area No of Rooms', 'Area Population']
for i, feat in enumerate(scatter_features):
    axes[i].scatter(df[feat], df['Price'], alpha=0.3, color=PALETTE[i], s=10)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Price')
    axes[i].set_title(f'{feat} vs Price')
    z = np.polyfit(df[feat], df['Price'], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df[feat])
    axes[i].plot(x_sorted, p(x_sorted), color='#B87333', linewidth=2, linestyle='--')
plt.suptitle('Relationships')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Price'], bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
plt.axvline(df['Price'].mean(), color=PALETTE[1], linestyle='--', linewidth=2)
plt.axvline(df['Price'].median(), color=PALETTE[2], linestyle='--', linewidth=2)
plt.xlabel('Price')
plt.ylabel('Count')
plt.title('Price Distribution')
plt.show()

In [ ]:
X = df[features]
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(rmse)
print(r2)

In [ ]:
model_unscaled = LinearRegression()
model_unscaled.fit(X_train, y_train)
print(model_unscaled.intercept_)
print(model_unscaled.coef_)

In [ ]:
os.makedirs('../model', exist_ok=True)
joblib.dump(model, '../model/house_price_model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
joblib.dump(features, '../model/feature_names.pkl')
joblib.dump({'rmse': rmse, 'r2': r2}, '../model/metrics.pkl')